In [ ]:
import os
import re
from pathlib import Path
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2


In [ ]:
# -------------------------------------------------------------------------
# 1. Dataset: Binary Lesion Mask (0 = BG/Healthy, 1 = Lesion) + Category Target
# -------------------------------------------------------------------------
class BinaryPlantSegDataset(Dataset):
    def __init__(self, img_dir, mask_dir, max_samples=5000, transform=None):
        self.img_dir = Path(img_dir)
        self.mask_dir = Path(mask_dir)
        self.transform = transform

        # Match stems regardless of .jpg vs .png extension
        img_stems = {
            f.stem: f for f in self.img_dir.glob("*.*") 
            if f.suffix.lower() in [".jpg", ".jpeg", ".png"]
        }
        mask_stems = {
            f.stem: f for f in self.mask_dir.glob("*.*") 
            if f.suffix.lower() in [".jpg", ".jpeg", ".png"]
        }
        
        common_stems = sorted(list(set(img_stems.keys()) & set(mask_stems.keys())))

        # Subsample to keep execution fast and within free GPU quota
        if max_samples and len(common_stems) > max_samples:
            np.random.seed(42)
            common_stems = list(np.random.choice(common_stems, max_samples, replace=False))

        self.samples = [(img_stems[s], mask_stems[s]) for s in common_stems]

        # Extract disease classes from filenames (e.g. 'apple_black_rot_1' -> 'apple_black_rot')
        raw_labels = [re.sub(r'_\d+$', '', s) for s in common_stems]
        unique_labels = sorted(list(set(raw_labels)))
        self.label_to_idx = {name: i for i, name in enumerate(unique_labels)}
        self.labels = [self.label_to_idx[l] for l in raw_labels]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, mask_path = self.samples[idx]

        # Read RGB image and raw multi-class annotation mask
        image = cv2.imread(str(img_path))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        raw_mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
        
        # Binarize: 0 remains 0 (BG/healthy), any disease ID (>0) becomes 1 (Lesion)
        binary_mask = (raw_mask > 0).astype(np.int64)

        if self.transform:
            augmented = self.transform(image=image, mask=binary_mask)
            image = augmented["image"]
            mask = augmented["mask"].long()
        else:
            mask = torch.tensor(binary_mask, dtype=torch.long)

        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return image, mask, label




In [ ]:

# =========================================================================
# 1. THE BRAIN: Dynamic Phase-Weighted Loss (DPW-Loss)
# =========================================================================
class DPWLoss(nn.Module):
    def __init__(self, total_epochs):
        super().__init__()
        self.total_epochs = total_epochs
        self.ce = nn.CrossEntropyLoss()
        
    def focal_loss(self, inputs, targets, alpha=0.25, gamma=2.0):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = alpha * (1 - pt) ** gamma * ce_loss
        return focal_loss.mean()

    def dice_loss(self, inputs, targets, smooth=1.0):
        # Convert targets to one-hot for Dice calculation
        preds = F.softmax(inputs, dim=1)
        targets_one_hot = F.one_hot(targets, num_classes=inputs.shape[1]).permute(0, 3, 1, 2).float()
        
        intersection = (preds * targets_one_hot).sum(dim=(2, 3))
        union = preds.sum(dim=(2, 3)) + targets_one_hot.sum(dim=(2, 3))
        dice = 1 - (2. * intersection + smooth) / (union + smooth)
        return dice.mean()

    def forward(self, inputs, targets, epoch):
        # Calculate individual losses
        l_ce = self.ce(inputs, targets)
        l_focal = self.focal_loss(inputs, targets)
        l_dice = self.dice_loss(inputs, targets)

        # Dynamic Phase Weighting based on epoch progress
        progress = epoch / self.total_epochs
        
        if progress < 0.33:
            # Phase 1: Foundational Learning (CE dominant)
            a, b, g = 1.0, 0.1, 0.1
        elif progress < 0.66:
            # Phase 2: Addressing Imbalance (Focal dominant)
            a, b, g = 0.1, 1.0, 0.1
        else:
            # Phase 3: Boundary Refinement (Dice dominant)
            a, b, g = 0.1, 0.5, 1.0

        return (a * l_ce) + (b * l_focal) + (g * l_dice)

# =========================================================================
# 2. THE BODY: RepMobileunit (Edge-Optimized Backbone Block)
# =========================================================================
class RepMobileunit(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.deploy = False
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.stride = stride

        # Training-time multi-branch topology
        # Branch 1: 3x3 Depthwise
        self.dw_3x3 = nn.Sequential(
            nn.Conv2d(in_channels, in_channels, 3, stride=stride, padding=1, groups=in_channels, bias=False),
            nn.BatchNorm2d(in_channels)
        )
        # Branch 2: 1x1 Depthwise
        self.dw_1x1 = nn.Sequential(
            nn.Conv2d(in_channels, in_channels, 1, stride=stride, padding=0, groups=in_channels, bias=False),
            nn.BatchNorm2d(in_channels)
        )
        # Branch 3: Identity (only if input/output match and stride is 1)
        self.identity = nn.BatchNorm2d(in_channels) if stride == 1 else None

        # Pointwise stage
        self.pw = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 1, bias=False),
            nn.BatchNorm2d(out_channels)
        )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        if self.deploy:
            # Inference mode: completely streamlined (implemented via self.reparameterize())
            return self.relu(self.pw(self.reparameterized_dw(x)))

        # Training mode: parallel branches
        dw_out = self.dw_3x3(x) + self.dw_1x1(x)
        if self.identity:
            dw_out += self.identity(x)
        
        return self.relu(self.pw(dw_out))
    
    def reparameterize(self):
        """
        Call this after training to mathematically fuse dw_3x3, dw_1x1, and identity 
        into a single self.reparameterized_dw layer, dropping parameters by ~80%.
        (Placeholder for the matrix algebra fusion logic).
        """
        self.deploy = True


In [ ]:
@torch.no_grad()
def evaluate_segmentation_metrics(model, dataloader, device, num_classes=2):
    model.eval()
    
    # Trackers for confusion matrix components
    total_tp = 0
    total_fp = 0
    total_fn = 0
    total_tn = 0
    
    # Class-wise intersection and union for mIoU
    intersection_per_class = torch.zeros(num_classes, device=device)
    union_per_class = torch.zeros(num_classes, device=device)

    for images, masks, _ in dataloader:
        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        seg_out, _ = model(images)
        preds = torch.argmax(seg_out, dim=1)  # (B, H, W)

        # Flatten tensors for pixel-level calculations
        preds_flat = preds.view(-1)
        masks_flat = masks.view(-1)

        # 1. Binary Lesion Metrics (focusing on foreground disease: Class 1)
        tp = ((preds_flat == 1) & (masks_flat == 1)).sum().item()
        fp = ((preds_flat == 1) & (masks_flat == 0)).sum().item()
        fn = ((preds_flat == 0) & (masks_flat == 1)).sum().item()
        tn = ((preds_flat == 0) & (masks_flat == 0)).sum().item()

        total_tp += tp
        total_fp += fp
        total_fn += fn
        total_tn += tn

        # 2. Multi-class IoU computation (accumulated across all classes)
        for cls in range(num_classes):
            pred_cls = (preds_flat == cls)
            target_cls = (masks_flat == cls)
            
            intersection = (pred_cls & target_cls).sum()
            union = (pred_cls | target_cls).sum()
            
            intersection_per_class[cls] += intersection
            union_per_class[cls] += union

    # Compute metrics with epsilon to prevent division by zero
    eps = 1e-7
    accuracy = (total_tp + total_tn) / (total_tp + total_tn + total_fp + total_fn + eps)
    precision = total_tp / (total_tp + total_fp + eps)
    recall = total_tp / (total_tp + total_fn + eps)
    f1 = 2 * (precision * recall) / (precision + recall + eps)
    
    # Calculate Mean IoU across all classes (0 = Background, 1 = Lesion)
    iou_per_class = (intersection_per_class + eps) / (union_per_class + eps)
    miou = iou_per_class.mean().item()

    print("\n" + "="*45)
    print("         VALIDATION SEGMENTATION METRICS      ")
    print("="*45)
    print(f"Accuracy  (Acc.) : {accuracy * 100:.2f}%")
    print(f"Precision (Prec.): {precision * 100:.2f}%")
    print(f"Recall    (Rec.) : {recall * 100:.2f}%")
    print(f"F1 Score  (F1)   : {f1:.4f}")
    print(f"Mean IoU  (mIoU) : {miou * 100:.2f}% (Lesion IoU: {iou_per_class[1].item() * 100:.2f}%)")
    print("="*45 + "\n")

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "miou": miou
    }

In [ ]:

# =========================================================================
# 3. THE NECK: HBAA Lite (Heterogeneous Feature Aggregation)
# =========================================================================
class StripPooling(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.pool_h = nn.AdaptiveAvgPool2d((None, 1))
        self.pool_w = nn.AdaptiveAvgPool2d((1, None))
        self.conv = nn.Conv2d(channels, channels, 1)

    def forward(self, x):
        h_str = self.pool_h(x)
        w_str = self.pool_w(x)
        # Expand and fuse
        out = h_str.expand_as(x) + w_str.expand_as(x)
        return torch.sigmoid(self.conv(out)) * x

class HBAALite(nn.Module):
    def __init__(self, channels):
        super().__init__()
        # Simulates Window Attention efficiently for edge devices
        # Option A: 3x3 depthwise convolution (identical receptive field balance)
        self.local_window = nn.Conv2d(
            channels, channels, kernel_size=3, padding=1, groups=channels, bias=False
        )       
        
        # Captures elongated lesions
        self.strip_pool = StripPooling(channels)
        self.fusion = nn.Conv2d(channels * 2, channels, 1)

    def forward(self, x):
        local_feat = self.local_window(x)
        strip_feat = self.strip_pool(x)
        concat = torch.cat([local_feat, strip_feat], dim=1)
        return self.fusion(concat) + x

# =========================================================================
# 4. THE FULL ARCHITECTURE
# =========================================================================
class CropDiseaseNet(nn.Module):
    def __init__(self, num_seg_classes=2, num_disease_classes=10):
        super().__init__()
        self.enc1 = RepMobileunit(3, 32, stride=2)   # (B, 32, 112, 112)
        self.enc2 = RepMobileunit(32, 64, stride=2)  # (B, 64, 56, 56)
        self.enc3 = RepMobileunit(64, 128, stride=2) # (B, 128, 28, 28)
        self.hbaa = HBAALite(128)
        
        # Step-by-step decoder with skip connections
        self.up1 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2) # 28 -> 56
        self.dec_conv1 = nn.Sequential(
            nn.Conv2d(64 + 64, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True)
        )
        
        self.up2 = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2)  # 56 -> 112
        self.dec_conv2 = nn.Sequential(
            nn.Conv2d(32 + 32, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True)
        )
        
        self.final_up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False) # 112 -> 224
        self.final_conv = nn.Conv2d(32, num_seg_classes, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(e1)
        e3 = self.enc3(e2)
        
        feat = self.hbaa(e3)
        
        d1 = self.up1(feat)
        d1 = torch.cat([d1, e2], dim=1)
        d1 = self.dec_conv1(d1)
        
        d2 = self.up2(d1)
        d2 = torch.cat([d2, e1], dim=1)
        d2 = self.dec_conv2(d2)
        
        seg_out = self.final_conv(self.final_up(d2))
        return seg_out

In [ ]:

# =========================================================================
# 5. DATASET & TRAINING LOOP UPDATES
# =========================================================================

train_transform = A.Compose([
    A.Resize(224, 224),
    A.HorizontalFlip(p=0.5),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])
# Validation transforms (no random flip, only deterministic resize and norm)
val_transform = A.Compose([
    A.Resize(224, 224),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

def train_hybrid_model():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    # Kaggle Train & Validation Paths
    train_img_folder = "/kaggle/input/datasets/weitianqi/plantseg/plantsegv2/images/train/"
    train_mask_folder = "/kaggle/input/datasets/weitianqi/plantseg/plantsegv2/annotations/train/"
    
    val_img_folder = "/kaggle/input/datasets/weitianqi/plantseg/plantsegv2/images/val/"
    val_mask_folder = "/kaggle/input/datasets/weitianqi/plantseg/plantsegv2/annotations/val/"

    # 1. Instantiate Train Dataset & Loader
    train_dataset = BinaryPlantSegDataset(
        img_dir=train_img_folder, 
        mask_dir=train_mask_folder, 
        max_samples=1000,
        transform=train_transform
    )
    
    train_loader = DataLoader(
        train_dataset,
        batch_size=32,
        shuffle=True,
        num_workers=2,
        pin_memory=True
    )

    # 2. Instantiate Validation Dataset & Loader
    val_dataset = BinaryPlantSegDataset(
        img_dir=val_img_folder, 
        mask_dir=val_mask_folder, 
        max_samples=200,             # Keep small for fast validation
        transform=val_transform
    )
    
    val_loader = DataLoader(
        val_dataset,
        batch_size=32,
        shuffle=False,
        num_workers=2,
        pin_memory=True
    )
    
    num_disease_classes = len(train_dataset.label_to_idx)
    epochs = 30

    model = CropDiseaseNet(
        num_seg_classes=2,
        num_disease_classes=num_disease_classes
    ).to(device)

    seg_criterion = DPWLoss(total_epochs=epochs)
    cls_criterion = nn.CrossEntropyLoss() 
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    scaler = torch.amp.GradScaler('cuda') if device.type == 'cuda' else None

    print("\nTraining Hybrid RepMobile + DPW-Loss Model...")
    
    for epoch in range(epochs):
        model.train()
        running_seg_loss = 0.0
        running_cls_loss = 0.0

        for images, masks, labels in train_loader:
            images = images.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            if device.type == 'cuda':
                with torch.amp.autocast('cuda'):
                    seg_out, cls_out = model(images)
                    loss_seg = seg_criterion(seg_out, masks, epoch)
                    loss_cls = cls_criterion(cls_out, labels)
                    # Downweight classification loss so gradients don't thrash segmentation features
                    total_loss = loss_seg + 0.05 * loss_cls

                scaler.scale(total_loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                seg_out, cls_out = model(images)
                loss_seg = seg_criterion(seg_out, masks, epoch)
                loss_cls = cls_criterion(cls_out, labels)
                total_loss = loss_seg + 0.05 * loss_cls
                total_loss.backward()
                optimizer.step()

            running_seg_loss += loss_seg.item()
            running_cls_loss += loss_cls.item()

        avg_seg = running_seg_loss / len(train_loader)
        avg_cls = running_cls_loss / len(train_loader)
        print(f"Epoch [{epoch+1:02d}/{epochs:02d}] - Seg Loss: {avg_seg:.4f} | Cls Loss: {avg_cls:.4f} | Total: {avg_seg + avg_cls:.4f}")
        
        # Validation metrics per epoch
        evaluate_segmentation_metrics(model, val_loader, device)

    # Save directly to Kaggle's working directory so weights persist
    torch.save(model.state_dict(), "/kaggle/working/hybrid_edge_disease_model.pth")
    print("\nSaved weights to '/kaggle/working/hybrid_edge_disease_model.pth'.")

In [ ]:
train_hybrid_model()